# This is the start of the coursework.

In [59]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### Data Preparation

### Loading and Inspecting Data
We first begin by importing the training dataset (`epl-training.csv`) into a DataFrame using Pandas. Using a DataFrame allows us to inspect, clean, and engineer features containing variables of different types for later modelling. 

We also add a `Date` column to our DataFrame so that Python can correctly interpret it as an actual date rather than a meaningless string. This will allow us to organise matches matches in chronological order and create season labels.

In [60]:
# Note for others - input your own path to the data below
data_path = "/users/ahmedelganady/Desktop/Uni/Year 3/COMP0036/Beat the Bookie/Data_Files/"

# Load provided training data file
train_df = pd.read_csv(
    data_path + "epl-training.csv",
    parse_dates=["Date"],
    dayfirst=True
)

# Load the 2025/26 season
train_df_extra = pd.read_csv(
    data_path + "E0.csv",
    parse_dates=["Date"],
    dayfirst=True
)

# Keep only columns up to AR - to match provided data's format
def keep_until_AR(df):
    cols = list(df.columns)
    if "AR" not in cols:
        raise ValueError(
            "Column 'AR' not found in train_df_extra. "
            "Check the E0.csv file is from football-data and is the correct format."
        )
    idx = cols.index("AR")
    return df.iloc[:, :idx + 1]

train_df_extra = keep_until_AR(train_df_extra)

# Drop division/time columns
cols_to_drop = [c for c in ["Div", "Time"] if c in train_df_extra.columns]
train_df_extra = train_df_extra.drop(columns=cols_to_drop, errors="ignore")

# Concatenate into train_df - so the rest of the notebook works unchanged
train_df = pd.concat([train_df, train_df_extra], ignore_index=True)

# Ensure rows needed for the GLM strength feature exist 
required_cols = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG"]
missing_required = [c for c in required_cols if c not in train_df.columns]
if missing_required:
    raise ValueError(f"Missing required columns after concat: {missing_required}")

train_df = train_df.dropna(subset=required_cols).reset_index(drop=True)

# Remove duplicates
dedup_keys = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG"]
train_df = train_df.drop_duplicates(subset=dedup_keys).reset_index(drop=True)

# Quick inspection 
print("train_df shape:", train_df.shape)
print("Date range:", train_df["Date"].min(), "→", train_df["Date"].max())


train_df shape: (9650, 22)
Date range: 2000-08-19 00:00:00 → 2025-12-08 00:00:00


### Sorting the Data in Chronological Order
The match data provided in (`epl-training.csv`) appears to already be sorted. However, I will sort them again as a precautionary practice.

In [61]:
# Sorts the whole DataFrame by the Date column
train_df.sort_values('Date', inplace=True)

# Resets the old indexing to the new indexing for the sorted data
train_df.reset_index(drop=True, inplace=True)

train_df.tail()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR
9645,2025-12-06,Everton,Nott'm Forest,3.0,0.0,H,2.0,0.0,H,C Kavanagh,...,5.0,3.0,6.0,3.0,17.0,15.0,4.0,3.0,0.0,0.0
9646,2025-12-06,Leeds,Liverpool,3.0,3.0,D,0.0,0.0,D,A Taylor,...,5.0,7.0,5.0,4.0,14.0,14.0,3.0,3.0,0.0,0.0
9647,2025-12-07,Fulham,Crystal Palace,1.0,2.0,A,1.0,1.0,D,S Barrott,...,4.0,5.0,6.0,4.0,5.0,8.0,0.0,0.0,0.0,0.0
9648,2025-12-07,Brighton,West Ham,1.0,1.0,D,0.0,0.0,D,S Hooper,...,4.0,5.0,10.0,6.0,12.0,8.0,2.0,2.0,0.0,0.0
9649,2025-12-08,Wolves,Man United,1.0,4.0,A,1.0,1.0,D,M Salisbury,...,2.0,10.0,1.0,9.0,17.0,12.0,3.0,2.0,0.0,0.0


### Creating a (Per Match) Goal Difference Feature
The Goal Difference feature will be defined as: 

<center><i>GD = FTHG - FTAG</i></center>

This will quantify how many more or less goals the home team was able to get by the away team. It is important to emphasize that this is a per-match feature and not to be confused with the conventional goal-difference aggregate feature in football league tables.

**Why is this feature useful?**

It will be used in later modelling features such as:
- Quickly interpreting a match's 'intensity'.
- Estimating a team's attack and defense strength in the Poisson GLM.
- Calculating performance a team's performance metrics.

In [62]:
train_df['GD'] = train_df['FTHG'] - train_df['FTAG']

train_df.tail()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,AST,HC,AC,HF,AF,HY,AY,HR,AR,GD
9645,2025-12-06,Everton,Nott'm Forest,3.0,0.0,H,2.0,0.0,H,C Kavanagh,...,3.0,6.0,3.0,17.0,15.0,4.0,3.0,0.0,0.0,3.0
9646,2025-12-06,Leeds,Liverpool,3.0,3.0,D,0.0,0.0,D,A Taylor,...,7.0,5.0,4.0,14.0,14.0,3.0,3.0,0.0,0.0,0.0
9647,2025-12-07,Fulham,Crystal Palace,1.0,2.0,A,1.0,1.0,D,S Barrott,...,5.0,6.0,4.0,5.0,8.0,0.0,0.0,0.0,0.0,-1.0
9648,2025-12-07,Brighton,West Ham,1.0,1.0,D,0.0,0.0,D,S Hooper,...,5.0,10.0,6.0,12.0,8.0,2.0,2.0,0.0,0.0,0.0
9649,2025-12-08,Wolves,Man United,1.0,4.0,A,1.0,1.0,D,M Salisbury,...,10.0,1.0,9.0,17.0,12.0,3.0,2.0,0.0,0.0,-3.0


### Creating a Season Indicator Feature

The data English Premier League provided spans 10+ years, it is important for this project that we can easliy identify which match belongs to which season. For eg: matches between August 2014 and May 2015 belong in the 14/15 season. 

**Why is this an important feature?**

This will allow us to later apply validation techniques such as *Leave One Season Out*, and without a label identifying which matches belong to which season matches from different seasons can be mixed up, affecting the validation process. 

**Season Indicator logic**

The simplest rule would be to say that if a match is in January – June → it belongs to the previous season and if it is in July – December → it belongs to the season starting that year.

In [63]:
# Create a function to assign season label based on match date - in accordance to feature logic explained above.

def season_name(date):
    month = date.month
    year = date.year
    
    if month >= 8:          
        return year 
    else:                   
        return year - 1
    
# Apply the function to DataFrame
train_df['Season'] = train_df['Date'].apply(season_name)

# Confirm
train_df[['Date', 'Season']].tail()

,Date,Season
9645,2025-12-06,2025
9646,2025-12-06,2025
9647,2025-12-07,2025
9648,2025-12-07,2025
9649,2025-12-08,2025


Run a few sanity checks to ensure everything is working properly.

In [64]:
# Check the total number of seasons
train_df['Season'].unique()

# Count the total number of matches per season 
# Note: each season should show 380 matches
train_df['Season'].value_counts().sort_index()


Season
2000    380
2001    380
2002    380
2003    380
2004    380
2005    380
2006    380
2007    380
2008    380
2009    380
2010    380
2011    380
2012    380
2013    380
2014    380
2015    380
2016    380
2017    380
2018    380
2019    380
2020    380
2021    380
2022    380
2023    380
2024    380
2025    150
Name: count, dtype: int64

## Constructing The Design Matrix For GLM

In this section I will represent the football matches numerically so that they can be used by the Generalised Linear Model (GLM). As this model is fully numerical, we can not give it strings.

**How will the matrix work?**

For every match:
- the home team will be assigned a '+1'.
- the away team will be assigned a '-1'.
- every other team will be assigned a '0'.

### Extract Team List & Create Index Mapping

In order to build the GLM model we must first identify all the unique teams that appear in our data set. We then create a dictionary which maps team names to their unique column indices - so that we can refer to them in the matrix.

Note: This matrix will include all the teams in the dataset, those that got relegated and promoted. Furthermore, I have checked if there is a team which is new to the EPL with no historical data in our dataset - there wasn't. 

In [65]:
# Extracting all team names from dataset
home_team_names = train_df['HomeTeam'].dropna().unique().tolist()

away_team_names = train_df['AwayTeam'].dropna().unique().tolist()

team_names = sorted(list(set(home_team_names+away_team_names))) # used the set function to remove duplicates 

# Create a Dictionary to map unique team names to unique indices
team_index = {}
i = 0

for team in team_names:
    team_index[team] = i
    i += 1

# Sanity Checks
print("Number of teams:", len(team_names))
team_index

teams_2526 = sorted(set(
    train_df.loc[train_df["Season"] == 2025, "HomeTeam"].tolist() +
    train_df.loc[train_df["Season"] == 2025, "AwayTeam"].tolist()
))

missing_in_mapping = [t for t in teams_2526 if t not in team_index]

print("Teams in 2025/26:", len(teams_2526))
print("Missing in team_index:", missing_in_mapping)

home_space = train_df["HomeTeam"][train_df["HomeTeam"].astype(str).str.match(r"^\s|\s$")].unique()
away_space = train_df["AwayTeam"][train_df["AwayTeam"].astype(str).str.match(r"^\s|\s$")].unique()

print("HomeTeam with edge whitespace:", home_space)
print("AwayTeam with edge whitespace:", away_space)

teams_hist = set(pd.read_csv(data_path + "epl-training.csv")["HomeTeam"].unique())
teams_new = set(train_df_extra["HomeTeam"].unique()) 

new_not_hist = sorted(list(teams_new - teams_hist))

print("Teams in E0.csv not in historical HomeTeam list:")
print(new_not_hist)

Number of teams: 46
Teams in 2025/26: 20
Missing in team_index: []
HomeTeam with edge whitespace: []
AwayTeam with edge whitespace: []
Teams in E0.csv not in historical HomeTeam list:
[]


### Build the GLM Design Matrix & Target Vector
In this section I will convert each football match into numbers, so that I can input them into the model later. We must create the Design Matrix, *X*, where each row represents one match and each column represents a unique team and the Target vector, *y*, which is a list of the goal differences for each match. Note: *X* and *y* should have the same number of rows.

**Design Matrix Logic:**

For every match:
- the home team will be assigned a '+1'.
- the away team will be assigned a '-1'.
- every other team will be assigned a '0'.

The GLM uses *X* and *y* together, and fits the model using a maximum likelihood estimation to calculate a strength value for each team across all matches. 

In [75]:
def glm_data(matches_df, team_names, team_index):
    
    # Removes any empty rows (otherise NaN error)
    matches_df = matches_df.dropna(subset=['HomeTeam', 'AwayTeam'])
    
    # No. of matches & teams
    no_matches = len(matches_df)
    no_teams = len(team_names)
    
    # Create an empty matrix, X
    X = np.zeros((no_matches, no_teams))

    # y is the GD column from the DataFrame
    y = matches_df['GD'].values

    # Ensure index is formatted properly (numbered in order)
    matches_df = matches_df.reset_index(drop=True)

    # Fill X row by row
    for m in range(no_matches):
        # Reads the home and away teams' names
        h_team = matches_df.loc[m, 'HomeTeam']
        a_team = matches_df.loc[m, 'AwayTeam']

        # Find the teams' index number
        h_col = team_index[h_team]
        a_col = team_index[a_team]

        # Sets a value of '+1' for home, and '-1' for away
        X[m, h_col] = 1.0
        X[m, a_col] = -1.0

    return X, y


# Sanity Check
X_all, y_all = glm_data(train_df, team_names, team_index)
print("Design matrix dimensions:", X_all.shape)
print("Target vector dimensions:", y_all.shape)


Design matrix dimensions: (9650, 46)
Target vector dimensions: (9650,)


In [92]:
def fit_weighted_glm_strengths(prev_df, team_names, team_index, ref_team, season=None, decay_factor=0.5):
    # Compute weights if season is provided (same logic you already wrote)
    if season is not None:
        prev_df = prev_df.copy()
        prev_df.loc[:, 'Season Age'] = season - prev_df['Season']
        prev_df.loc[:, 'Weight'] = decay_factor ** prev_df['Season Age']
        weights = prev_df['Weight'].values
    else:
        weights = None

    # Build design matrix
    X_prev, y_prev = glm_data(prev_df, team_names, team_index)

    # Remove reference column
    ref_index = team_index[ref_team]
    X_prev_red = np.delete(X_prev, ref_index, axis=1)

    # Add intercept
    X_prev_glm = sm.add_constant(X_prev_red)

    # Fit GLM (with weights if available)
    if weights is None:
        model_prev = sm.GLM(y_prev, X_prev_glm, family=sm.families.Gaussian())
    else:
        model_prev = sm.GLM(y_prev, X_prev_glm, family=sm.families.Gaussian(), freq_weights=weights)

    results_prev = model_prev.fit()
    params_prev = results_prev.params
    teamStrength_coefs_prev = params_prev[1:]  # intercept is params_prev[0]

    # Build full team-strength dict including ref team
    season_strengths = {}
    pos_coef = 0
    for t in team_names:
        if t == ref_team:
            season_strengths[t] = 0.0
        else:
            season_strengths[t] = teamStrength_coefs_prev[pos_coef]
            pos_coef += 1

    return season_strengths

In [93]:
def fit_glm_strengths_upto(train_df, cutoff_date, team_names, team_index, ref_team):
    
    df_past = train_df[train_df["Date"] < cutoff_date].copy()

    if len(df_past) == 0:
        return {t: 0.0 for t in team_names}

    X_past, y_past = glm_data(df_past, team_names, team_index)

    ref_index = team_index[ref_team]
    X_past_red = np.delete(X_past, ref_index, axis=1)

    X_past_glm = sm.add_constant(X_past_red)

    model = sm.GLM(y_past, X_past_glm, family=sm.families.Gaussian())
    results = model.fit()

    params = results.params
    coefs = params[1:]  # params[0] is intercept

    strengths = {}
    pos = 0
    for t in team_names:
        if t == ref_team:
            strengths[t] = 0.0
        else:
            strengths[t] = coefs[pos]
            pos += 1

    return strengths


### Identifiability

This is a unique problem of this feature. Recall that the GLM equation is given by:

$$
\theta_{\text{home}} - \theta_{\text{away}}
$$

The model can only understand the differences between strengths, so any values which keep the differences the same between two teams are equally valid - giving the model infinite strength solutions. 

Example:

Consider teams A & B:
- Strength(A) = 1.5
- Strength(B) = 0.5
- GD = 1.0

Now add '+10' to both A & B's strengths:
- Strength(A) = 11.5
- Strength(B) = 10.5
- GD = 1.0 (stays the same)

These are both equally valid solutions - so how can the model determine which solution is the correct one? The solution to this problem is to choose one team (ideally a mid table team) and set its strength value to equal '0', this becomes the reference team and all other teams' strengths are measured relative to this team's strength. This allows the model to have one unique strength solution per team.

In [94]:
# Choose any team as reference - choosing last team in DataFrame for ease
ref_team = team_names[-1]
print("The Chosen Reference team with strength = 0 is:", ref_team)

# Determine its column index
ref_index = team_index[ref_team]

# Remove that column from our design matrix, X
X_new = np.delete(X_all, ref_index, axis=1)

# Sanity Check
print("Original X dimensions:", X_all.shape)
print("New X dimensions:", X_new.shape)


The Chosen Reference team with strength = 0 is: Wolves
Original X dimensions: (9650, 46)
New X dimensions: (9650, 45)


## Finding a Strength Value For Each Team

This is done by:
- Fitting a linear regression model
- Translating the model's coefficients to team strength values

### Fitting the GLM

So far, I have created a design matrix, *X*, and a goal differences vector, *y*, so we can fit a simple linear regression model to obtain each team's strength.
<br></br>
<center>Predicted GD ≈ $ \theta_{\text{home}} - \theta_{\text{away}} + b$ </center>

where:
- $\theta_{\text{home}}$ is the home team's strength value
- $\theta_{\text{away}}$ is the away team's strength value
- $b$ is the y-intercept of the linear regression model

The model uses maximum likelihood estimation to choose values for all the teams' strengths so that the predicted GD value is as close as possible to the real GD in the data provided.  

We use a Gaussian GLM here because GD values can be estimated with a normal distribution curve over many matches.

In [95]:
# Add an intercept to the reduced design matrix - this is just a column of contants
X_glm = sm.add_constant(X_new)

# Fit a GLM 
model = sm.GLM(y_all, X_glm, family=sm.families.Gaussian())

# Store the results 
results = model.fit()

# Sanity Check - no. of parameters learned by the model
print("Number of parameters learned:", len(results.params))
print(results.params[:10])


Number of parameters learned: 46
[ 0.35357379  1.298688    0.34649682  0.17623979  0.30510664 -0.10642042
  0.20510664  0.08539154 -0.58269585  0.52404082]


### Assigning Strength Values to Each Team
In the previous step we fit the GLM, the model was able to learn a list of numbers called the *'parameters'*. The first number in that list is the intercept, and the rest of the numbers correspond to the team strength coefficients in order of index, except for the reference team which has been extracted. This team's strength is set as *'0'*. So, a team with a positive strength value is predicted to be stronger than the reference team and a team with a negative value is predicted to be weaker.

So what we need to do is extract the intercept value from the list and assign the remaining parameters (strength coefficients) to their respective teams and store this in a new DataFrame. By the end we should have one strength value per team.

In [96]:
# Defining the intercept parameters from the fitted model
parameters = results.params
intercept = parameters[0]
strength_c = parameters[1:]

# Quick Sanity check
print("Number of team strength coefficients:", len(strength_c))
print("Number of team colums in the reduced design matrix:", X_new.shape[1]) # Both values should be equal

# Create an empty dictionary to place all the teams and strength coefficients in
team_strengths = {}
c_position = 0              # This is to track the position inside the full list of teams

for i in team_names:
    if i == ref_team:
        team_strengths[i] = 0.0
    else: 
        team_strengths[i] = strength_c[c_position]
        c_position += 1

#  Another Sanity Check 3dzzza
print("Number of teams:", len(team_names))
print("Number of strength values:", len(team_strengths))

# Now we convert this dictionary into a DataFrame
teamStrengths_df = pd.DataFrame({
    'Team': list(team_strengths.keys()),
    'Strength Coefficient': list(team_strengths.values())
})

# Sort the teams from highest strength values (strongest) to lowest strength value (weakest)
teamStrengths_df = teamStrengths_df.sort_values('Strength Coefficient', ascending=False).reset_index(drop=True)

# Quick Sanity Check
print(teamStrengths_df)

Number of team strength coefficients: 45
Number of team colums in the reduced design matrix: 45
Number of teams: 46
Number of strength values: 46
                Team  Strength Coefficient
0            Arsenal              1.298688
1           Man City              1.287448
2          Liverpool              1.278580
3            Chelsea              1.268578
4         Man United              1.241435
5          Tottenham              0.797235
6          Brentford              0.524041
7            Everton              0.471120
8          Newcastle              0.384324
9        Aston Villa              0.346497
10          Brighton              0.340907
11         Leicester              0.324657
12         Blackburn              0.305107
13             Leeds              0.250484
14     Middlesbrough              0.248730
15    Crystal Palace              0.243177
16          West Ham              0.231762
17            Bolton              0.205107
18            Fulham              0.1

## Validation

In the previous section I calculated a single strength value for each team using the entire dataset. Although, this gives a good estimate of the overall team strength, it has a flaw. It uses future matches to describe past matches - this is known as a *'data leakage'*.

For example: if we let the model use results from the 2023/24 season to predict match outcomes of the 2010/11 season, we would be letting the model "see the future" before predicting the present. This is means the model basically cheated by peeking into future results to predict a present match. This would give inflated accuracy numbers as we validate the model, but will fail once we test it on future data (Jan 2026 matches).

Hence, to build a more realistic model, we must ensure that every match only uses match data that was available to it up that point, and nothing beyond it. We do this be generating a feature called *'Time Aware Team Strengths'*. We do this by:

- We sort all the match data by date (and season). (Already Done)
- For a given season, $S_{k}$, we calculate the team strength values using only the seasons before it.
- We then assign this strength, $S_{k}$, to all the matches in that season, $S_{k}$.
- For the first season in the dataset we assign neutral strength values of 0.

This methodology ensures that each match in the dataset is described using only the match data that would have been available up to that point, which should increase predictive accuracy for gameweek 24 (Jan 31st). 

Furthermore, I have decided to weigh team strengths based on recency. This will use the same methodology as before, but it will assign higher weights to more recent seasons, and lower weights to older seasons. I will do this by applying an exponential decay factor of 0.8 per season so that recent seasons exert a greater influence on the GLM fit.

I should then compare models; one which use this recency bias and one which doesn't and evaluate differences in predictive accuracy between the models to determine whether this would improve accuracy or not.

In [97]:
# Obtain a sorted set of all the seasons
# First remove any empty rows
train_df = train_df.dropna(subset=['Season']).reset_index(drop=True)
seasons = sorted(train_df['Season'].unique())

# Create an empty list which will store all team strength values for different seasons
all_strengths = []
decay_factor = 0.8

# Create a for loop to loop through each season one by one

for i, season in enumerate(seasons):
    print("\n\nProcessing season:", season)
    
    prev_seasons = seasons[:i]
    print("Previous season data which is used for this season:", prev_seasons)
    
    # Case 1: No previous data - season 2000
    if not prev_seasons:
        print("There is no past data available for this season, so will assign '0.0' strength value to all teams.")
        for team in team_names:
            all_strengths.append({
                'Season': season,
                'Team': team,
                'Strength Coefficient': 0.0
            })
        continue
    # Case 2: Previous data available - all other seasons
    else:
        prev_df = train_df[train_df['Season'].isin(prev_seasons)].copy()
        print("No. of previous matches used to estimate strength values:", len(prev_df))
    
        # Computing recency-bias by adding higher weights to recent seasons
        # Find the age of the data compared to current season
        prev_df.loc[:, 'Season Age'] = season - prev_df['Season']

        # Calculate the weight of that season
        prev_df.loc[:, 'Weight'] = decay_factor ** prev_df['Season Age']
    
        # Build the design matrix for previous seasons
        X_prev, y_prev = glm_data(prev_df, team_names, team_index)
        print("X_prev dimensions:", X_prev.shape, "\ny_prev length:", len(y_prev))
    
    # Arrange the weights in a vector of same dimensions as y
    weights = prev_df['Weight'].values
    
    # Remove the reference team's column
    ref_index = team_index[ref_team]
    X_prev_red = np.delete(X_prev, ref_index, axis=1)
    print("X_prev_red dimensions:", X_prev_red.shape)
    X_prev_glm = sm.add_constant(X_prev_red)
    
    # Fit the GLM with weights
    model_prev = sm.GLM(
        y_prev,
        X_prev_glm,
        family = sm.families.Gaussian(),
        freq_weights = weights 
    )
    results_prev = model_prev.fit()
    

    # Obtain parameters
    params_prev = results_prev.params
    intercept_prev = params_prev[0]
    teamStrength_coefs_prev = params_prev[1:]
    
    print("Intercept value for previous season's model:", intercept_prev)
    print("No. of team strength coefficients obtained:", len(teamStrength_coefs_prev))
    
    # Redetermine the strengths for all the teams this season
    season_strengths = {}
    pos_coef = 0
    
    for t in team_names:
        if t == ref_team:
            season_strengths[t] = 0.0
        else:
            season_strengths[t] = teamStrength_coefs_prev[pos_coef]
            pos_coef += 1
    
    # Store one row per (Season, Team)
    for t in team_names:
        all_strengths.append({
            'Season': season,
            'Team': t,
            'Strength Coefficient': season_strengths[t]
        })

season_strengths_df = pd.DataFrame(all_strengths)
season_strengths_df[season_strengths_df["Season"] == 2025].head()



Processing season: 2000
Previous season data which is used for this season: []
There is no past data available for this season, so will assign '0.0' strength value to all teams.


Processing season: 2001
Previous season data which is used for this season: [2000]
No. of previous matches used to estimate strength values: 380
X_prev dimensions: (380, 46) 
y_prev length: 380
X_prev_red dimensions: (380, 45)
Intercept value for previous season's model: 0.4789473684210529
No. of team strength coefficients obtained: 45


Processing season: 2002
Previous season data which is used for this season: [2000, 2001]
No. of previous matches used to estimate strength values: 760
X_prev dimensions: (760, 46) 
y_prev length: 760
X_prev_red dimensions: (760, 45)
Intercept value for previous season's model: 0.37807017543859645
No. of team strength coefficients obtained: 45


Processing season: 2003
Previous season data which is used for this season: [2000, 2001, 2002]
No. of previous matches used to esti

Intercept value for previous season's model: 0.28556249680525847
No. of team strength coefficients obtained: 45


Processing season: 2022
Previous season data which is used for this season: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
No. of previous matches used to estimate strength values: 8360
X_prev dimensions: (8360, 46) 
y_prev length: 8360
X_prev_red dimensions: (8360, 45)
Intercept value for previous season's model: 0.26991347541559596
No. of team strength coefficients obtained: 45


Processing season: 2023
Previous season data which is used for this season: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
No. of previous matches used to estimate strength values: 8740
X_prev dimensions: (8740, 46) 
y_prev length: 8740
X_prev_red dimensions: (8740, 45)
Intercept value for previous season's model: 0.29926

,Season,Team,Strength Coefficient
1150,2025,Arsenal,1.146765
1151,2025,Aston Villa,0.340111
1152,2025,Birmingham,-0.097034
1153,2025,Blackburn,-0.072901
1154,2025,Blackpool,-0.287930


In [98]:
# Add the time-based team-strengths to the main DataFrame

for col in ['HomeTeamStrength', 'AwayTeamStrength', 'StrengthDifference']:
    if col in train_df.columns:
        del train_df[col]

# Convert the all_strengths list into a DataFrame
season_strengths_df = pd.DataFrame(all_strengths)

# Create a home team's strength table - rename the columns
home_strengths_df = season_strengths_df.rename(
    columns={
        'Team': 'HomeTeam',
        'Strength Coefficient': 'HomeTeamStrength'
    }
)

# Add home team strength values into the Data Frame
train_df = train_df.merge(
    home_strengths_df,
    on=['Season', 'HomeTeam'],
    how='left'
)

# Create an away team's strength table - rename the columns
away_strengths_df = season_strengths_df.rename(
    columns={
        'Team': 'AwayTeam',
        'Strength Coefficient': 'AwayTeamStrength'
    }
)

# Add away team strength values into the Data Frame
train_df = train_df.merge(
    away_strengths_df,
    on=['Season', 'AwayTeam'],
    how='left'
)

# Calculate the Strength Difference between the home and away teams
train_df['StrengthDifference'] = train_df['HomeTeamStrength'] - train_df['AwayTeamStrength']

# 7. Sanity checks
print("\nMerged training data with time-aware strengths:")
display(train_df[['Season', 'Date', 'HomeTeam', 'AwayTeam',
                  'HomeTeamStrength', 'AwayTeamStrength', 'StrengthDifference']])

print("\nMissing values check:")
print(train_df[['HomeTeamStrength', 'AwayTeamStrength']].isna().sum())






Merged training data with time-aware strengths:


,Season,Date,HomeTeam,AwayTeam,HomeTeamStrength,AwayTeamStrength,StrengthDifference
0,2000,2000-08-19,Charlton,Man City,0.000000,0.000000,0.000000
1,2000,2000-08-19,Chelsea,West Ham,0.000000,0.000000,0.000000
2,2000,2000-08-19,Coventry,Middlesbrough,0.000000,0.000000,0.000000
3,2000,2000-08-19,Derby,Southampton,0.000000,0.000000,0.000000
4,2000,2000-08-19,Leeds,Everton,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...
9645,2025,2025-12-06,Everton,Nott'm Forest,0.139683,0.144159,-0.004476
9646,2025,2025-12-06,Leeds,Liverpool,-0.216959,1.361383,-1.578342
9647,2025,2025-12-07,Fulham,Crystal Palace,0.103796,0.159136,-0.055340
9648,2025,2025-12-07,Brighton,West Ham,0.294150,0.110716,0.183434



Missing values check:
HomeTeamStrength    0
AwayTeamStrength    0
dtype: int64


In [99]:
season = 2025
prev_seasons = [s for s in seasons if s < season]

prev_df = train_df[train_df['Season'].isin(prev_seasons)].copy()
in_season_df = train_df[train_df['Season'] == season].copy()
prev_df = pd.concat([prev_df, in_season_df], ignore_index=True)

prev_df["Season Age"] = season - prev_df["Season"]
prev_df["Weight"] = decay_factor ** prev_df["Season Age"]

total_weight = prev_df["Weight"].sum()
weight_2025 = prev_df.loc[prev_df["Season"] == 2025, "Weight"].sum()

print("Share of total weight from 2025/26:", weight_2025 / total_weight)


Share of total weight from 2025/26: 0.09013027778126306


Now we rename some of the variables to remain consistent with the brief's naming convention.

In [100]:
train_df["HomeStrength"] = train_df["HomeTeamStrength"]
train_df["AwayStrength"] = train_df["AwayTeamStrength"]
train_df["StrengthDiff"] = train_df["HomeStrength"] - train_df["AwayStrength"]

# Sanity Check
train_df[["HomeTeam", "AwayTeam", "HomeStrength", "AwayStrength", "StrengthDiff"]].head()

,HomeTeam,AwayTeam,HomeStrength,AwayStrength,StrengthDiff
0,Charlton,Man City,0.0,0.0,0.0
1,Chelsea,West Ham,0.0,0.0,0.0
2,Coventry,Middlesbrough,0.0,0.0,0.0
3,Derby,Southampton,0.0,0.0,0.0
4,Leeds,Everton,0.0,0.0,0.0


Must check that there are no empty cells (NaN) before passing the data into the model.

In [101]:
# Check if there are any NaNs - How many?
print("No. of NaNs:")
print(train_df[["HomeStrength", "AwayStrength", "StrengthDiff"]].isna().sum())

# Nice - no NaNs

No. of NaNs:
HomeStrength    0
AwayStrength    0
StrengthDiff    0
dtype: int64


## Applying the Time-Based Team Strengths to the Test Data

In this section I will apply the GLM team strength values to the unseen matches. Firstly, I will load the *'epl-test.csv'* file and apply a proper datetime format to the Date column so that it can be comprehended by python. I will then use the season function from before to assign each match to a season.

Then, I will take merge the team strength values that were obtained above onto this data set. I will do this for both the *Home* and *Away* teams - giving every match its own *HomeStrength* and *AwayStrength* columns.

Then, I will create Strength Difference (*StrengthDiff = HomeStrength – AwayStrength*), like I did for the training dataset. The test DataFrame should now be ready for prediction and will contain the 3 final strength columns which the ML model will need.

In [107]:
# Adding the team strength values to the test data (leakage-free, as-of-date)

data_path = "/users/ahmedelganady/Desktop/Uni/Year 3/COMP0036/Beat the Bookie/Data_Files/"
test_df = pd.read_csv(data_path + "epl-test.csv")

# Parse date + season label
test_df["Date"] = pd.to_datetime(test_df["Date"], format="%d %b %y")
test_df["Season"] = test_df["Date"].apply(season_name)

# Optional: keep ONLY if the Forest name mismatch still exists
forest_name_change = {"Nottingham Forest": "Nott'm Forest"}
test_df["HomeTeam"] = test_df["HomeTeam"].replace(forest_name_change)
test_df["AwayTeam"] = test_df["AwayTeam"].replace(forest_name_change)

# --- Leakage-free strength assignment for test_df (as-of-date) ---

strength_cache = {}
unique_dates = sorted(test_df["Date"].dropna().unique())

for d in unique_dates:
    strength_cache[d] = fit_glm_strengths_upto(
        train_df, d, team_names, team_index, ref_team
    )

test_df["HomeStrength"] = test_df.apply(
    lambda r: strength_cache[r["Date"]].get(r["HomeTeam"], 0.0),
    axis=1
)
test_df["AwayStrength"] = test_df.apply(
    lambda r: strength_cache[r["Date"]].get(r["AwayTeam"], 0.0),
    axis=1
)

test_df["StrengthDiff"] = test_df["HomeStrength"] - test_df["AwayStrength"]

# Sanity checks
display(
    test_df[[
        "Date", "HomeTeam", "AwayTeam",
        "Season", "HomeStrength", "AwayStrength", "StrengthDiff"
    ]].head(10)
)
print("Missing strengths:", test_df[["HomeStrength","AwayStrength"]].isna().sum())
print("Unique test dates:", len(unique_dates), "| GLM fits cached:", len(strength_cache))


,Date,HomeTeam,AwayTeam,Season,HomeStrength,AwayStrength,StrengthDiff
0,2026-01-31,Leeds,Arsenal,2025,0.250484,1.298688,-1.048204
1,2026-01-31,Liverpool,Newcastle,2025,1.278580,0.384324,0.894256
2,2026-01-31,Tottenham,Man City,2025,0.797235,1.287448,-0.490213
3,2026-01-31,Wolves,Bournemouth,2025,0.000000,0.085392,-0.085392
4,2026-01-31,Aston Villa,Brentford,2025,0.346497,0.524041,-0.177544
5,2026-01-31,Brighton,Everton,2025,0.340907,0.471120,-0.130212
6,2026-01-31,Chelsea,West Ham,2025,1.268578,0.231762,1.036816
7,2026-01-31,Man United,Fulham,2025,1.241435,0.189282,1.052153
8,2026-01-31,Sunderland,Burnley,2025,-0.022444,-0.091555,0.069111
9,2026-01-31,Nott'm Forest,Crystal Palace,2025,0.167536,0.243177,-0.075641


Missing strengths: HomeStrength    0
AwayStrength    0
dtype: int64
Unique test dates: 1 | GLM fits cached: 1


**Note:** The 2025/26 season is defined as 2025 earlier on. Not to be confused with the previous season - 2024/25 (which is *season = 2024*).

In [109]:
d0 = test_df["Date"].min()
print("Earliest test date:", d0)
print("Latest training match used:", train_df[train_df["Date"] < d0]["Date"].max())

Earliest test date: 2026-01-31 00:00:00
Latest training match used: 2025-12-08 00:00:00


In [114]:
print("train_df shape:", train_df.shape)
print("train_df columns contain Season/Date?:", "Season" in train_df.columns, "Date" in train_df.columns)
print("Season min/max:", train_df["Season"].min(), train_df["Season"].max())
print("Count season==2025:", (train_df["Season"] == 2025).sum())
print("Date dtype:", train_df["Date"].dtype)
print("Non-null dates in season 2025:", ((train_df["Season"] == 2025) & train_df["Date"].notna()).sum())

train_df[train_df["Season"] == 2025][["Date","HomeTeam","AwayTeam","FTHG","FTAG"]].tail()


train_df shape: (9650, 30)
train_df columns contain Season/Date?: True True
Season min/max: 2000 2025
Count season==2025: 150
Date dtype: datetime64[ns]
Non-null dates in season 2025: 150


,Date,HomeTeam,AwayTeam,FTHG,FTAG
9645,2025-12-06,Everton,Nott'm Forest,3.0,0.0
9646,2025-12-06,Leeds,Liverpool,3.0,3.0
9647,2025-12-07,Fulham,Crystal Palace,1.0,2.0
9648,2025-12-07,Brighton,West Ham,1.0,1.0
9649,2025-12-08,Wolves,Man United,1.0,4.0


In [118]:
# H-minimum: leakage-free, date-based strengths for in-season matches (2025/26 only)

# Ensure these columns don't already exist to avoid merge confusion
for col in ["HomeStrength", "AwayStrength", "StrengthDiff"]:
    if col in train_df.columns:
        del train_df[col]

# Start by using your existing season-level columns if you already merged them
# (these are HomeTeamStrength/AwayTeamStrength/StrengthDifference)
# We'll create final columns HomeStrength/AwayStrength/StrengthDiff consistently.

# Default: use your season-level strengths (works for seasons <= 2024)
if "HomeTeamStrength" in train_df.columns and "AwayTeamStrength" in train_df.columns:
    train_df["HomeStrength"] = train_df["HomeTeamStrength"]
    train_df["AwayStrength"] = train_df["AwayTeamStrength"]
    train_df["StrengthDiff"] = train_df["HomeStrength"] - train_df["AwayStrength"]
else:
    # If you didn't merge season strengths, set neutral defaults first
    train_df["HomeStrength"] = 0.0
    train_df["AwayStrength"] = 0.0
    train_df["StrengthDiff"] = 0.0

# Now overwrite ONLY season 2025 matches with leakage-free date-based strengths
mask_2526 = (train_df["Season"] == 2025) & (train_df["Date"].notna())
df_2526 = train_df.loc[mask_2526, ["Date", "HomeTeam", "AwayTeam"]].copy()

# Cache strengths per unique match date within 2025/26
date_cache = {}
unique_train_dates = sorted(df_2526["Date"].unique())

for d in unique_train_dates:
    date_cache[d] = fit_glm_strengths_upto(train_df, d, team_names, team_index, ref_team)

# Assign
train_df.loc[mask_2526, "HomeStrength"] = df_2526.apply(
    lambda r: date_cache[r["Date"]].get(r["HomeTeam"], 0.0),
    axis=1
).values

train_df.loc[mask_2526, "AwayStrength"] = df_2526.apply(
    lambda r: date_cache[r["Date"]].get(r["AwayTeam"], 0.0),
    axis=1
).values

train_df.loc[mask_2526, "StrengthDiff"] = (
    train_df.loc[mask_2526, "HomeStrength"] - train_df.loc[mask_2526, "AwayStrength"]
)

# Sanity checks
print("2025/26 rows updated:", mask_2526.sum())
print("Missing strengths in 2025/26:", train_df.loc[mask_2526, ["HomeStrength","AwayStrength"]].isna().sum())
display(train_df.loc[mask_2526, ["Date","HomeTeam","AwayTeam","HomeStrength","AwayStrength","StrengthDiff"]].tail(10))


2025/26 rows updated: 150
Missing strengths in 2025/26: HomeStrength    0
AwayStrength    0
dtype: int64


,Date,HomeTeam,AwayTeam,HomeStrength,AwayStrength,StrengthDiff
9640,2025-12-06,Newcastle,Burnley,0.379138,-0.096022,0.475161
9641,2025-12-06,Man City,Sunderland,1.281050,-0.025301,1.306352
9642,2025-12-06,Tottenham,Brentford,0.790838,0.527961,0.262877
9643,2025-12-06,Bournemouth,Chelsea,0.078217,1.264386,-1.186170
9644,2025-12-06,Aston Villa,Arsenal,0.339711,1.295246,-0.955535
9645,2025-12-06,Everton,Nott'm Forest,0.463765,0.181230,0.282535
9646,2025-12-06,Leeds,Liverpool,0.243275,1.274223,-1.030947
9647,2025-12-07,Fulham,Crystal Palace,0.186302,0.235904,-0.049602
9648,2025-12-07,Brighton,West Ham,0.337575,0.226410,0.111165
9649,2025-12-08,Wolves,Man United,0.000000,1.234553,-1.234553


In [119]:
train_df.loc[train_df["Season"] == 2025, ["Date","HomeTeam","AwayTeam","HomeStrength","AwayStrength","StrengthDiff"]] \
       .sort_values("Date") \
       .head(10)

train_df.loc[train_df["Season"] == 2025, ["Date","HomeTeam","AwayTeam","HomeStrength","AwayStrength","StrengthDiff"]] \
       .sort_values("Date") \
       .tail(10)

d = train_df.loc[train_df["Season"] == 2025, "Date"].min()
print("Earliest 25/26 match date:", d)
print("Latest match used to fit that date:", train_df[train_df["Date"] < d]["Date"].max())


Earliest 25/26 match date: 2025-08-15 00:00:00
Latest match used to fit that date: 2025-05-25 00:00:00
